# Custom CNN — GTSRB Traffic Sign Classification

**Owner:** Naweedh
**Model:** Custom CNN (built from scratch, no pretrained weights)
**Framework:** PyTorch

This notebook trains and evaluates the group's from-scratch baseline. It uses the shared split in
`data/processed/gtsrb_data.npz` (made by `src/data_split.py`: seed 42, 64×64 RGB, values in [0, 1]),
so its results compare directly with ResNet50, MobileNetV3 and EfficientNetB0.

**Runs on:** Google Colab (GPU) or locally (CUDA / Apple-Silicon MPS / CPU). The setup cell detects which.

| Section | Contents |
|---|---|
| 0 | Environment setup (Colab or local) |
| 1 | Imports, config, seeding |
| 2 | Load shared split |
| 3 | Preprocessing & augmentation |
| 4 | Model architecture + design rationale |
| 5 | Training (train/val only) |
| 6 | Learning curves |
| 7 | Final test-set evaluation |
| 8 | Error analysis |
| 9 | Efficiency (params, size, inference time) |
| 10 | Save config & metrics |
| 11 | Notes for the report |

## 0. Environment setup

- **Colab:** Runtime → Change runtime type → **T4 GPU**. The cell below mounts Google Drive and uses
  `MyDrive/DL_GroupProject` (it clones the repo there if the folder doesn't exist). Put `gtsrb_data.npz` in
  `MyDrive/DL_GroupProject/data/processed/`. Outputs are written to Drive, so they survive a runtime reset.
- **Local:** run the notebook from the `notebooks/` folder or from the repo root.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/DL_GroupProject")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/AaqibAR/DL_GroupProject.git", str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "data" / "processed" / "gtsrb_data.npz"
RESULTS_DIR = ROOT / "results" / "custom_cnn"
CONFIG_DIR = ROOT / "configs"
CKPT_DIR = ROOT / "checkpoints"          # gitignored — weights are not committed
for d in (RESULTS_DIR, CONFIG_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Colab:", IN_COLAB)
print("Repo root:", ROOT)
print("Data file exists:", DATA_PATH.exists())

## 1. Imports, config, seeding

In [ ]:
import json
import time
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    confusion_matrix, classification_report,
)

print("PyTorch:", torch.__version__)

In [ ]:
# All tunable settings live here and are saved to configs/custom_cnn.json at the end.
# Shared settings (seed, image size, split) must match src/data_split.py; don't change them.
CONFIG = {
    "model_name": "Custom CNN",
    # --- shared across the group ---
    "seed": 42,
    "img_size": 64,
    "num_classes": 43,
    # --- architecture ---
    "conv_channels": [32, 64, 128],   # one entry per conv block (2 conv layers per block)
    "conv_dropout": 0.2,
    "dense_units": 256,
    "fc_dropout": 0.5,
    # --- training ---
    "batch_size": 64,
    "epochs": 40,                     # upper bound; early stopping usually ends sooner
    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "lr_scheduler": "ReduceLROnPlateau(val_loss, factor=0.5, patience=3)",
    "early_stopping_patience": 8,     # epochs without val-loss improvement
    "label_smoothing": 0.0,
    "use_class_weights": False,       # ablation: True = weight loss by inverse class frequency
    # --- augmentation (train set only) ---
    "augment": True,
    "aug_rotation_deg": 10,
    "aug_translate": 0.1,
    "aug_scale": [0.9, 1.1],
    "aug_brightness": 0.3,
    "aug_contrast": 0.3,
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DEVICE_NAME = f"Apple MPS ({platform.machine()})"
else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = f"CPU ({platform.processor() or platform.machine()})"
CONFIG["device"] = DEVICE_NAME
print("Device:", DEVICE_NAME)

In [ ]:
# Official GTSRB class names (index = ClassId), used for readable error analysis
CLASS_NAMES = [
    "Speed limit (20km/h)", "Speed limit (30km/h)", "Speed limit (50km/h)", "Speed limit (60km/h)",
    "Speed limit (70km/h)", "Speed limit (80km/h)", "End of speed limit (80km/h)", "Speed limit (100km/h)",
    "Speed limit (120km/h)", "No passing", "No passing for vehicles over 3.5t",
    "Right-of-way at next intersection", "Priority road", "Yield", "Stop", "No vehicles",
    "Vehicles over 3.5t prohibited", "No entry", "General caution", "Dangerous curve left",
    "Dangerous curve right", "Double curve", "Bumpy road", "Slippery road", "Road narrows on the right",
    "Road work", "Traffic signals", "Pedestrians", "Children crossing", "Bicycles crossing",
    "Beware of ice/snow", "Wild animals crossing", "End of all speed and passing limits",
    "Turn right ahead", "Turn left ahead", "Ahead only", "Go straight or right", "Go straight or left",
    "Keep right", "Keep left", "Roundabout mandatory", "End of no passing",
    "End of no passing by vehicles over 3.5t",
]
assert len(CLASS_NAMES) == CONFIG["num_classes"]

## 2. Load the shared split

Everyone loads the same `.npz`, so all four models see identical train/val/test images.
**The test arrays are not used until Section 7.**

In [ ]:
data = np.load(DATA_PATH)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"dtype={X_train.dtype}, range=[{X_train.min():.2f}, {X_train.max():.2f}]")

counts = np.bincount(y_train, minlength=CONFIG["num_classes"])
print(f"Train class counts: min={counts.min()} (class {counts.argmin()}), "
      f"max={counts.max()} (class {counts.argmax()}), imbalance ratio={counts.max() / counts.min():.1f}")

## 3. Preprocessing & augmentation

- **Standardisation:** the per-channel mean and std come from the **training set only** (no leakage)
  and are applied to train, val and test.
- **Augmentation (train only):** small rotations, shifts, scaling and brightness/contrast jitter. These imitate
  the viewpoint and lighting changes in real dashcam frames.
- **No horizontal or vertical flips:** flipping changes what many signs mean (*Turn left* ↔ *Turn right*,
  *Keep left* ↔ *Keep right*, *Dangerous curve left* ↔ *right*), so it would create wrongly labelled images.

In [ ]:
CHANNEL_MEAN = X_train.mean(axis=(0, 1, 2)).tolist()
CHANNEL_STD = X_train.std(axis=(0, 1, 2)).tolist()
CONFIG["channel_mean"], CONFIG["channel_std"] = CHANNEL_MEAN, CHANNEL_STD
print("Train mean:", np.round(CHANNEL_MEAN, 4), "std:", np.round(CHANNEL_STD, 4))

normalize = v2.Normalize(mean=CHANNEL_MEAN, std=CHANNEL_STD)

if CONFIG["augment"]:
    train_transform = v2.Compose([
        v2.RandomAffine(
            degrees=CONFIG["aug_rotation_deg"],
            translate=(CONFIG["aug_translate"], CONFIG["aug_translate"]),
            scale=tuple(CONFIG["aug_scale"]),
        ),
        v2.ColorJitter(brightness=CONFIG["aug_brightness"], contrast=CONFIG["aug_contrast"]),
        normalize,
    ])
else:
    train_transform = normalize
eval_transform = normalize


class GTSRBDataset(Dataset):
    # Wraps the NHWC float arrays from the .npz as CHW tensors for PyTorch
    def __init__(self, X, y, transform=None):
        self.X = torch.from_numpy(X).permute(0, 3, 1, 2)
        self.y = torch.from_numpy(y).long()
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = self.X[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, self.y[idx]


# Worker processes don't work well with notebook-defined classes on macOS, so only use them on CUDA
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
PIN = DEVICE.type == "cuda"
g = torch.Generator().manual_seed(CONFIG["seed"])

train_loader = DataLoader(GTSRBDataset(X_train, y_train, train_transform), batch_size=CONFIG["batch_size"],
                          shuffle=True, generator=g, num_workers=NUM_WORKERS, pin_memory=PIN)
val_loader = DataLoader(GTSRBDataset(X_val, y_val, eval_transform), batch_size=256,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN)
test_loader = DataLoader(GTSRBDataset(X_test, y_test, eval_transform), batch_size=256,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN)
print(f"Batches/epoch: train={len(train_loader)} val={len(val_loader)}")

In [ ]:
# Visual check: the same training image with 7 different random augmentations (useful figure for the report)
aug_only = v2.Compose([t for t in train_transform.transforms if not isinstance(t, v2.Normalize)]) \
    if CONFIG["augment"] else (lambda x: x)
rng = np.random.default_rng(CONFIG["seed"])
fig, axes = plt.subplots(4, 8, figsize=(14, 7.5))
for row, idx in zip(axes, rng.choice(len(X_train), 4, replace=False)):
    original = torch.from_numpy(X_train[idx]).permute(2, 0, 1)
    row[0].imshow(X_train[idx])
    row[0].set_title("original", fontsize=8)
    for ax in row[1:]:
        ax.imshow(aug_only(original).clamp(0, 1).permute(1, 2, 0).numpy())
    row[0].set_ylabel(CLASS_NAMES[y_train[idx]][:22], fontsize=7)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Training augmentation examples")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "augmentation_examples.png", dpi=150)
plt.show()

## 4. Model architecture

```
Input 3×64×64
 └─ Block 1: [Conv3×3(32) → BN → ReLU] ×2 → MaxPool 2×2 → Dropout2d   → 32×32×32
 └─ Block 2: [Conv3×3(64) → BN → ReLU] ×2 → MaxPool 2×2 → Dropout2d   → 64×16×16
 └─ Block 3: [Conv3×3(128) → BN → ReLU] ×2 → MaxPool 2×2 → Dropout2d  → 128×8×8
 └─ Flatten (8192) → Dense(256) → BN → ReLU → Dropout(0.5) → Dense(43 logits)
```

**Design rationale** (for the report):
- **Stacked 3×3 convolutions** (VGG-style): two 3×3 layers see the same 5×5 area as one 5×5 layer, but with
  fewer parameters and an extra non-linearity.
- **Channels double (32→64→128) as the image halves:** early layers pick up edges and colours, deeper layers
  pick up shapes, digits and symbols.
- **3 pooling stages:** 64→8 px, which is still enough to tell apart small details like "30" vs "80".
- **Batch normalisation:** makes training steadier and allows a higher learning rate. The conv layers have no
  bias because BN already adds one.
- **ReLU** in hidden layers. The model outputs raw logits and `CrossEntropyLoss` applies softmax internally.
- **Dropout** (light in the conv blocks, 0.5 before the classifier) plus weight decay limit overfitting. Most
  parameters are in the 8192→256 dense layer, so that is where overfitting is most likely.
- **No pretrained weights:** this is the from-scratch baseline that shows what transfer learning adds.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x):
        return self.block(x)


class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes, img_size, conv_channels, conv_dropout, dense_units, fc_dropout):
        super().__init__()
        blocks, in_ch = [], 3
        for out_ch in conv_channels:
            blocks.append(ConvBlock(in_ch, out_ch, conv_dropout))
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)

        spatial = img_size // (2 ** len(conv_channels))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_ch * spatial * spatial, dense_units, bias=False),
            nn.BatchNorm1d(dense_units),
            nn.ReLU(inplace=True),
            nn.Dropout(fc_dropout),
            nn.Linear(dense_units, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model():
    return TrafficSignCNN(
        num_classes=CONFIG["num_classes"], img_size=CONFIG["img_size"],
        conv_channels=CONFIG["conv_channels"], conv_dropout=CONFIG["conv_dropout"],
        dense_units=CONFIG["dense_units"], fc_dropout=CONFIG["fc_dropout"],
    ).to(DEVICE)


model = build_model()
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal parameters: {n_params:,}  (trainable: {n_trainable:,})")

# Per-layer parameter breakdown (useful for the complexity discussion)
layer_params = pd.DataFrame(
    [(name, tuple(p.shape), p.numel()) for name, p in model.named_parameters()],
    columns=["parameter", "shape", "count"],
)
layer_params["% of total"] = (100 * layer_params["count"] / n_params).round(2)
layer_params.sort_values("count", ascending=False).head(8)

## 5. Training

- **Loss:** categorical cross-entropy. Optional inverse-frequency class weights (`use_class_weights`) test
  whether the 10× class imbalance hurts the rare classes.
- **Optimizer:** AdamW (Adam with decoupled weight decay) with lr 1e-3.
- **LR schedule:** ReduceLROnPlateau halves the learning rate when val loss stops improving for 3 epochs.
- **Early stopping:** training ends after 8 epochs without val-loss improvement. The best-val-loss weights
  are kept as the final model.
- **Model selection uses validation data only.** The test set is not used here.

In [ ]:
if CONFIG["use_class_weights"]:
    class_weights = torch.tensor(len(y_train) / (CONFIG["num_classes"] * counts), dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=CONFIG["label_smoothing"])
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


def run_epoch(model, loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * yb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            n += yb.size(0)
    return total_loss / n, correct / n

In [ ]:
CKPT_PATH = CKPT_DIR / "custom_cnn_best.pt"
history = []
best_val_loss, best_epoch, epochs_no_improve = float("inf"), 0, 0

train_start = time.perf_counter()
for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.perf_counter()
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc = run_epoch(model, val_loader, train=False)
    lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    epoch_time = time.perf_counter() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_loss": val_loss, "val_acc": val_acc, "lr": lr, "epoch_time_s": epoch_time})

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss, best_epoch, epochs_no_improve = val_loss, epoch, 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f} | lr {lr:.1e} | {epoch_time:.1f}s"
          + ("  *best*" if improved else ""))

    if epochs_no_improve >= CONFIG["early_stopping_patience"]:
        print(f"Early stopping: no val-loss improvement for {epochs_no_improve} epochs.")
        break

train_time_s = time.perf_counter() - train_start
history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / "history.csv", index=False)
print(f"\nTraining time: {train_time_s / 60:.1f} min | epochs run: {len(history)} | best epoch: {best_epoch}")

## 6. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(history_df.epoch, history_df.train_loss, label="train")
axes[0].plot(history_df.epoch, history_df.val_loss, label="validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch")
axes[1].plot(history_df.epoch, history_df.train_acc, label="train")
axes[1].plot(history_df.epoch, history_df.val_acc, label="validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch")
axes[2].plot(history_df.epoch, history_df.lr, color="gray")
axes[2].set_yscale("log"); axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch")
for ax in axes[:2]:
    ax.axvline(best_epoch, ls="--", c="k", lw=0.8, label=f"best epoch ({best_epoch})")
    ax.legend()
plt.suptitle("Custom CNN — training history")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "learning_curves.png", dpi=150)
plt.show()

best_row = history_df.loc[history_df.epoch == best_epoch].iloc[0]
print(f"Best epoch {best_epoch}: train acc {best_row.train_acc:.4f}, val acc {best_row.val_acc:.4f}, "
      f"gap {best_row.train_acc - best_row.val_acc:+.4f}")

## 7. Final test-set evaluation

This is the **first and only** time the test set is used. The best-val-loss checkpoint is reloaded before
predicting. Metrics are macro-averaged (each of the 43 classes counts equally, which shows weak rare classes)
with weighted averages alongside. ROC-AUC is one-vs-rest.

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

all_probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        all_probs.append(torch.softmax(model(xb.to(DEVICE)), dim=1).cpu())
test_probs = torch.cat(all_probs).numpy()
test_pred = test_probs.argmax(1)

acc = accuracy_score(y_test, test_pred)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_test, test_pred, average="macro", zero_division=0)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_test, test_pred, average="weighted", zero_division=0)
roc_auc = roc_auc_score(y_test, test_probs, multi_class="ovr", average="macro",
                        labels=list(range(CONFIG["num_classes"])))

metrics = {
    "accuracy": acc,
    "precision_macro": p_macro, "recall_macro": r_macro, "f1_macro": f1_macro,
    "precision_weighted": p_w, "recall_weighted": r_w, "f1_weighted": f1_w,
    "roc_auc_ovr_macro": roc_auc,
    "n_test": int(len(y_test)), "n_misclassified": int((test_pred != y_test).sum()),
}
pd.Series(metrics).to_frame("Custom CNN")

In [ ]:
# Per-class precision / recall / F1
report = classification_report(y_test, test_pred, labels=list(range(CONFIG["num_classes"])),
                               target_names=CLASS_NAMES, output_dict=True, zero_division=0)
per_class = pd.DataFrame({k: report[k] for k in CLASS_NAMES}).T
per_class.index.name = "class"
per_class.insert(0, "class_id", range(CONFIG["num_classes"]))
per_class["train_count"] = counts
per_class.to_csv(RESULTS_DIR / "per_class_metrics.csv")
print("Worst 10 classes by F1:")
per_class.sort_values("f1-score").head(10)[["class_id", "precision", "recall", "f1-score", "support", "train_count"]]

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=list(range(CONFIG["num_classes"])))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(14, 12))
sns.heatmap(cm_norm, cmap="Blues", vmin=0, vmax=1, square=True, cbar_kws={"label": "fraction of true class"})
plt.xlabel("Predicted class"); plt.ylabel("True class")
plt.title(f"Custom CNN — normalised confusion matrix (test acc {acc:.4f})")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()
np.savetxt(RESULTS_DIR / "confusion_matrix.csv", cm, fmt="%d", delimiter=",")

## 8. Error analysis

Which signs does the model confuse, and what do the mistakes look like? This feeds the critical-analysis
section of the report.

In [ ]:
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
pairs = [(i, j, off_diag[i, j]) for i, j in zip(*np.nonzero(off_diag))]
confused = pd.DataFrame(pairs, columns=["true_id", "pred_id", "count"]).sort_values("count", ascending=False).head(15)
confused["true"] = confused.true_id.map(lambda i: CLASS_NAMES[i])
confused["predicted"] = confused.pred_id.map(lambda i: CLASS_NAMES[i])
confused["% of true class"] = (100 * confused["count"] / cm.sum(axis=1)[confused.true_id]).round(2)
confused.to_csv(RESULTS_DIR / "top_confusions.csv", index=False)
confused[["true", "predicted", "count", "% of true class"]]

In [ ]:
# Grid of misclassified test images, showing true label, predicted label and confidence
wrong = np.flatnonzero(test_pred != y_test)
show = np.random.default_rng(CONFIG["seed"]).choice(wrong, min(24, len(wrong)), replace=False) if len(wrong) else []
if len(show):
    fig, axes = plt.subplots(3, 8, figsize=(16, 7))
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, show):
        ax.imshow(X_test[i])
        ax.set_title(f"T: {CLASS_NAMES[y_test[i]][:18]}\nP: {CLASS_NAMES[test_pred[i]][:18]}\n"
                     f"conf {test_probs[i, test_pred[i]]:.2f}", fontsize=7)
    plt.suptitle("Custom CNN — misclassified test images")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "misclassified_examples.png", dpi=150)
    plt.show()

# Is the model confident when it's wrong? (calibration hint)
conf = test_probs.max(1)
print(f"Mean confidence — correct: {conf[test_pred == y_test].mean():.3f}, "
      f"wrong: {conf[test_pred != y_test].mean() if len(wrong) else float('nan'):.3f}")

In [ ]:
# Does training-set frequency explain per-class performance? (tests the class-imbalance hypothesis)
plt.figure(figsize=(7, 5))
plt.scatter(per_class.train_count, per_class["f1-score"])
for _, r in per_class.nsmallest(6, "f1-score").iterrows():
    plt.annotate(int(r.class_id), (r.train_count, r["f1-score"]), fontsize=8, xytext=(3, 3), textcoords="offset points")
plt.xlabel("Training images in class"); plt.ylabel("Test F1")
plt.title("Per-class F1 vs. training frequency")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "f1_vs_class_frequency.png", dpi=150)
plt.show()
print("Spearman correlation:", round(per_class[["train_count", "f1-score"]].corr(method="spearman").iloc[0, 1], 3))

## 9. Efficiency

In [ ]:
model_size_mb = CKPT_PATH.stat().st_size / 1024 ** 2


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def time_inference(batch_size, reps=50):
    x = torch.randn(batch_size, 3, CONFIG["img_size"], CONFIG["img_size"], device=DEVICE)
    with torch.no_grad():
        for _ in range(10):          # warm-up
            model(x)
        sync()
        t0 = time.perf_counter()
        for _ in range(reps):
            model(x)
        sync()
    return (time.perf_counter() - t0) / reps / batch_size * 1000   # ms per image


latency_ms = time_inference(1)
throughput_ms = time_inference(256)
efficiency = {
    "total_params": n_params,
    "trainable_params": n_trainable,
    "model_size_mb": round(model_size_mb, 2),
    "train_time_min": round(train_time_s / 60, 2),
    "epochs_run": len(history),
    "best_epoch": best_epoch,
    "mean_epoch_time_s": round(history_df.epoch_time_s.mean(), 2),
    "inference_ms_per_image_bs1": round(latency_ms, 3),
    "inference_ms_per_image_bs256": round(throughput_ms, 4),
    "device": DEVICE_NAME,
}
pd.Series(efficiency).to_frame("Custom CNN")

## 10. Save config & metrics

- `configs/custom_cnn.json`: every hyperparameter, for reproducibility.
- `results/custom_cnn/metrics.json`: one row for the group's model-comparison table.

In [ ]:
with open(CONFIG_DIR / "custom_cnn.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

summary = {"model": CONFIG["model_name"], **{k: round(float(v), 4) if isinstance(v, float) else v
                                            for k, v in metrics.items()}, **efficiency}
with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
for p in sorted([CONFIG_DIR / "custom_cnn.json", *RESULTS_DIR.iterdir()]):
    print("  ", p.relative_to(ROOT))

## 11. Notes for the report (fill in after training)

- **Overall result:** test accuracy ___, macro-F1 ___, ROC-AUC ___.
- **Fit:** train/val gap at the best epoch is ___. Overfitting, underfitting or a good fit? Point to the learning curves.
- **Most-confused pairs:** ___ (for example, speed limits with similar digits). Why? Low resolution, similar shape and colour.
- **Class imbalance:** Spearman(train count, F1) = ___. Did the rare classes do worse? Try `use_class_weights=True` and compare.
- **Augmentation:** try `augment=False` and compare val accuracy and the overfitting gap.
- **Efficiency:** ___ params, ___ MB, ___ ms/image. Compare with the pretrained models.
- **vs. transfer learning:** how close does a from-scratch model get to ResNet50, MobileNetV3 and EfficientNetB0?